# 99 — التقرير النهائي

بيجمع نتايج كل الـ notebooks (من `notebooks/reports/*.json`) في لوحة واحدة. شغّل الـ notebooks من 01 لـ 07 الأول، وبعدين الـ notebook ده.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd())); sys.path.insert(0, str(pathlib.Path.cwd() / "notebooks"))
from nb_utils import *
import matplotlib.pyplot as plt
files = sorted(REPORTS.glob("*.json"))
data = [json.loads(f.read_text(encoding="utf-8")) for f in files if f.stem != "summary"]
rows = [dict(notebook=d["notebook"], **r) for d in data for r in d["rows"]]
ALL = pd.DataFrame(rows)
print(f"{len(data)} notebook reports loaded:", [d["notebook"] for d in data])

In [ ]:
ORDER = ["PASS", "WARN", "SKIP", "FAIL", "ERROR"]
COL = {"PASS": "#59a14f", "WARN": "#edc948", "SKIP": "#bab0ab", "FAIL": "#e15759", "ERROR": "#9c2f2f"}
P = ALL.pivot_table(index="notebook", columns="status", values="test", aggfunc="count", fill_value=0).reindex(columns=ORDER, fill_value=0)
P["total"] = P.sum(axis=1); P["pass_rate_%"] = (100 * P["PASS"] / (P["PASS"] + P["FAIL"] + P["ERROR"]).replace(0, float("nan"))).round(0)
display(P)
P[ORDER].plot.barh(stacked=True, color=[COL[c] for c in ORDER], figsize=(10, 0.6 * len(P) + 1.5), title="Results per notebook"); plt.tight_layout(); plt.show()
tot = ALL.status.value_counts().reindex(ORDER, fill_value=0)
scored = tot["PASS"] + tot["FAIL"] + tot["ERROR"]
display(HTML(f"<h2>Overall: {tot['PASS']}/{scored} passed ({100 * tot['PASS'] / max(1, scored):.0f}%) &nbsp;|&nbsp; {tot['WARN']} warnings &nbsp;|&nbsp; {tot['SKIP']} skipped &nbsp;|&nbsp; {tot['FAIL'] + tot['ERROR']} failures</h2>"))

## ❌ الفشل (لازم يتصلّح)

In [ ]:
F = ALL[ALL.status.isin(["FAIL", "ERROR"])][["notebook", "category", "test", "status", "details"]]
display(F if len(F) else HTML("<b style='color:green'>no failures 🎉</b>"))

## ⚠️ التحذيرات (قرارات / تحسينات)

In [ ]:
W = ALL[ALL.status == "WARN"][["notebook", "category", "test", "details"]]
display(W if len(W) else HTML("<i>no warnings</i>"))

## ⏭️ الاختبارات اللي اتخطّت (محتاجة ربط / adapter)

In [ ]:
K = ALL[ALL.status == "SKIP"][["notebook", "test", "details"]]
display(K if len(K) else HTML("<i>nothing skipped</i>"))

## حسب الفئة

In [ ]:
C = ALL.pivot_table(index="category", columns="status", values="test", aggfunc="count", fill_value=0).reindex(columns=ORDER, fill_value=0)
display(C.sort_values("FAIL", ascending=False))
ALL.to_csv(REPORTS / "summary.csv", index=False, encoding="utf-8-sig")
(REPORTS / "summary.html").write_text(ALL.to_html(index=False, escape=True), encoding="utf-8")
print("saved:", REPORTS / "summary.csv", "and summary.html")